# 07 — Khảo sát hệ số phạt chung của QAOA

Sau mốc A1: thay P, giữ p=1, số phép đo, maxiter, seed và hai điểm khởi tạo. Đây là khảo sát A=B=P; hai hệ số độc lập chưa được triển khai. Dùng cùng mô hình và Aer của notebook 06. Phần giải thích tiếng Anh ở cuối notebook.

## Bước 1 — Kiểm tra môi trường và nạp mã đã có

Dùng `src/qaoa_experiment.py` bản Aer đã bàn giao. Không cần cài thêm thư viện so với notebook 05. Nếu đã sửa module, khởi động lại kernel trước khi import.

Bảng phiên bản xác định môi trường sinh kết quả. Cùng seed nhưng khác bộ mô phỏng/phiên bản không bắt buộc cho số đo giống nhau.

In [1]:
import gc
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import scipy
import qiskit
import qiskit_aer
import qiskit_optimization

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "qaoa_experiment.py").is_file()
             and (p / "data" / "instances").is_dir()), None)
assert ROOT is not None, "Mở notebook bên trong repo nt409-task-placement."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.qubo import build_qubo
from src.qaoa_experiment import (
    prepare_reference, probe_qaoa_resources,
    run_qaoa_experiment, save_experiment_report,
)

print("Python:", sys.version.split()[0])
print("Qiskit / Aer / Optimization:",
      qiskit.__version__, qiskit_aer.__version__, qiskit_optimization.__version__)
print("Repo:", ROOT)
print("RAM khả dụng (GiB):", round(psutil.virtual_memory().available / 2**30, 2))
assert sys.version_info[:2] == (3, 12), "Chọn kernel Conda Python 3.12 của dự án."
from time import perf_counter
from src.penalty_analysis import penalty_diagnostics

Python: 3.12.14
Qiskit / Aer / Optimization: 2.5.2 0.17.2 0.7.0
Repo: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
RAM khả dụng (GiB): 13.3


## Bước 2 — Chốt ma trận khảo sát

P = 1, 2, 3, 5, 8. Với hai instance, hai khởi tạo và ba seed, có 60 lượt. P=3 là đối chứng mới trong cùng đợt; không gộp các lượt A1 vào bảng mới. p0_reference chạy riêng với 30 lượt. Không đổi các yếu tố khác để rồi quy kết cải thiện cho P.

In [2]:
# RUN_INSTANCE_IDS = ["p1_tight", "p1_choice"]
RUN_INSTANCE_IDS = ["p0_reference"]
RESUME_MANIFEST = None
PENALTIES = [1, 2, 3, 5, 8]
SEEDS = [42, 43, 44]
INITIAL_POINTS = {"goc_cu": [0.3, 0.7], "gamma_nho": [0.3, 0.05]}
BASE_CONFIG = {
    "shots": 1024, "reps": 1, "maxiter": 35,
    "max_qubits": 23, "max_seconds": 300,
    "aer_threads": 4, "aer_memory_mb": 2048, "min_available_gib": 2.0,
}
# RESUME_MANIFEST = None
# Tiếp tục sau restart: đặt đường dẫn chỉ mục phase4a_penalty_runs cũ.
ALLOWED_IDS = {"p1_tight", "p1_choice", "p1_infeasible", "p0_reference"}
assert RUN_INSTANCE_IDS and len(set(RUN_INSTANCE_IDS)) == len(RUN_INSTANCE_IDS)
assert set(RUN_INSTANCE_IDS) <= ALLOWED_IDS
assert PENALTIES and len(set(PENALTIES)) == len(PENALTIES)
assert all(type(p) is int and p > 0 for p in PENALTIES)
assert SEEDS and len(set(SEEDS)) == len(SEEDS)
assert all(type(seed) is int and seed >= 0 for seed in SEEDS)
assert BASE_CONFIG["reps"] == 1, "Khảo sát này giữ p=1."
assert INITIAL_POINTS and all(len(v) == 2 and np.isfinite(v).all() for v in INITIAL_POINTS.values())
instances = {name: json.loads((ROOT / "data" / "instances" / f"{name}.json").read_text(encoding="utf-8")) for name in RUN_INSTANCE_IDS}
assert all(item["instance_id"] == name for name, item in instances.items())
def canonical_hash(value):
    text = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()
bundles = {(name, p): build_qubo(item, penalty=p) for name, item in instances.items() for p in PENALTIES}
references = {key: prepare_reference(bundle) for key, bundle in bundles.items()}
DESIGN = {
    "instance_ids": RUN_INSTANCE_IDS, "penalties": PENALTIES, "seeds": SEEDS,
    "initial_points": INITIAL_POINTS, "base_config": BASE_CONFIG, "diagnostics_version": 1,
    "instance_sha256": {name: canonical_hash(item) for name, item in instances.items()},
}
DESIGN_HASH = canonical_hash(DESIGN)
planned = []
for name, p, init_id, seed in product(RUN_INSTANCE_IDS, PENALTIES, INITIAL_POINTS, SEEDS):
    config = {**BASE_CONFIG, "seed": seed, "initial_point": INITIAL_POINTS[init_id].copy()}
    key = canonical_hash({"instance_sha256": DESIGN["instance_sha256"][name], "penalty": p,
                          "initial_point_id": init_id, "config": config, "diagnostics_version": 1})
    planned.append({"run_key": key, "instance_id": name, "penalty": p,
                    "initial_point_id": init_id, "seed": seed, "config": config})
print("Thiết kế:", DESIGN_HASH, "| Số lượt:", len(planned))
display(pd.DataFrame([{k: row[k] for k in ["instance_id", "penalty", "initial_point_id", "seed"]} for row in planned]))

Thiết kế: 20ab1a6868b175481b7465e5d7476c58b613b8ea23c33489a9e1ae85a6052f30 | Số lượt: 30


,instance_id,penalty,initial_point_id,seed
0,p0_reference,1,goc_cu,42
1,p0_reference,1,goc_cu,43
2,p0_reference,1,goc_cu,44
3,p0_reference,1,gamma_nho,42
4,p0_reference,1,gamma_nho,43
5,p0_reference,1,gamma_nho,44
6,p0_reference,2,goc_cu,42
7,p0_reference,2,goc_cu,43
8,p0_reference,2,goc_cu,44
9,p0_reference,2,gamma_nho,42


## Bước 3 — Kiểm chứng mô hình ở từng P

Không suy ra ground state mới từ P=3. Đối chiếu nghiệm placement gốc và cực tiểu QUBO riêng ở mỗi P. Với dữ liệu nguyên và hai host, P>2 là điều kiện đủ để cực tiểu toàn cục hợp lệ nếu có nghiệm; P=1 hoặc 2 cần xem tham chiếu, chưa có bảo đảm từ điều kiện đó.

In [3]:
size_rows = []
for (name, p), bundle in bundles.items():
    nbits = bundle["qubo"].get_num_vars()
    assert nbits <= BASE_CONFIG["max_qubits"]
    ref = references[(name, p)]
    size_rows.append({
        "Bộ dữ liệu": name, "P": p, "Mọi cực tiểu QUBO hợp lệ": ref["all_qubo_minimizers_feasible"], "Task": bundle["n"], "Host": bundle["m"],
        "Qubit": nbits, "Một mảng statevector (MiB)": 16 * 2**nbits / 2**20,
        "Có nghiệm gốc": ref["instance_feasible"],
        "Số host tối ưu gốc": ref["original_best_objective"],
        "Năng lượng QUBO nhỏ nhất": ref["minimum_energy"],
        "Hợp lệ khi lấy mẫu đều (%)": 100 * ref["uniform_baseline"]["feasible_probability"],
    })
display(pd.DataFrame(size_rows))

,Bộ dữ liệu,P,Mọi cực tiểu QUBO hợp lệ,Task,Host,Qubit,Một mảng statevector (MiB),Có nghiệm gốc,Số host tối ưu gốc,Năng lượng QUBO nhỏ nhất,Hợp lệ khi lấy mẫu đều (%)
0,p0_reference,1,True,4,2,23,128.0,True,2,2.0,0.097656
1,p0_reference,2,True,4,2,23,128.0,True,2,2.0,0.097656
2,p0_reference,3,True,4,2,23,128.0,True,2,2.0,0.097656
3,p0_reference,5,True,4,2,23,128.0,True,2,2.0,0.097656
4,p0_reference,8,True,4,2,23,128.0,True,2,2.0,0.097656


## Bước 4 — Đo thử chi phí trước khi tối ưu

Mỗi instance lấy một mạch tại góc đầu của cấu hình đầu tiên. Đây là chẩn đoán tài nguyên, **không tính thành một lượt QAOA đã tối ưu** và không lưu như kết quả.

Ước tính thời gian = thời gian lấy mẫu × (`maxiter` + 1), chưa gồm mọi chi phí và không phải bảo đảm. Xem bảng trước khi chạy Bước 6. Nếu ước tính gần/vượt ngân sách hoặc RAM khả dụng thấp, dừng để điều chỉnh nguồn lực/thiết kế.

Aer chạy tuần tự, tối đa 4 luồng cho một mạch. Giới hạn 2048 MiB chỉ áp cho lưu trạng thái của Aer, không giới hạn cứng toàn bộ Python. RAM đỉnh là đỉnh toàn vòng đời tiến trình, có thể gồm các cell trước.

In [4]:
probes = {}
for key, bundle in bundles.items():
    name, p = key
    trial = next(row for row in planned if row["instance_id"] == name and row["penalty"] == p)
    print("Đo thử:", name, "P=", p, flush=True)
    probes[key] = probe_qaoa_resources(bundle, trial["config"])
display(pd.DataFrame([{
    "Bộ dữ liệu": name, "P": p, "Qubit": item["num_qubits"],
    "Một lần lấy mẫu (s)": item["sampling_seconds"],
    "Ước tính phần lấy mẫu/lượt (s)": item["rough_maxiter_plus_final_sampling_seconds"],
    "RAM đỉnh Python (MiB)": item["resources"]["process_peak_rss_mib"],
} for (name, p), item in probes.items()]))
rough_campaign = sum(probes[(row["instance_id"], row["penalty"])]["rough_maxiter_plus_final_sampling_seconds"] for row in planned)
print("Ước tính riêng phần lấy mẫu (phút):", round(rough_campaign / 60, 2))
if any(p["rough_maxiter_plus_final_sampling_seconds"] >= BASE_CONFIG["max_seconds"] for p in probes.values()):
    raise RuntimeError("Đo thử chạm ngân sách: xem tài nguyên trước khi chạy Bước 6.")

Đo thử: p0_reference P= 1
Đo thử: p0_reference P= 2
Đo thử: p0_reference P= 3
Đo thử: p0_reference P= 5
Đo thử: p0_reference P= 8


,Bộ dữ liệu,P,Qubit,Một lần lấy mẫu (s),Ước tính phần lấy mẫu/lượt (s),RAM đỉnh Python (MiB)
0,p0_reference,1,23,0.540522,19.458803,300.226562
1,p0_reference,2,23,0.517455,18.628394,301.761719
2,p0_reference,3,23,0.499274,17.973871,302.187500
3,p0_reference,5,23,0.549471,19.780963,302.437500
4,p0_reference,8,23,0.498414,17.942886,302.707031


Ước tính riêng phần lấy mẫu (phút): 9.38


## Bước 5 — Chuẩn bị chỉ mục hoặc tiếp tục đợt đã có

Một JSON lượt chạy là một thí nghiệm. File `campaign_...json` chỉ mô tả thiết kế, các lượt hoàn tất và đường dẫn; không phải lượt bổ sung.

Chạy lại **riêng cell Bước 6** sẽ bỏ qua những lượt đã hoàn tất trong đợt hiện tại. Sau khi khởi động lại kernel, dùng `RESUME_MANIFEST` ở Bước 2 để tiếp tục đúng đợt. Đầu vào/cấu hình khác sẽ bị từ chối để tránh trộn các thiết kế.

Chỉ mục được cập nhật ngay sau mỗi lượt hoàn tất. Lượt chưa hoàn tất không có JSON kết quả riêng; số lượt còn lại là “chưa có báo cáo hoàn tất”, không tự diễn giải là vô nghiệm hay đã chạy thất bại.

In [5]:
if RESUME_MANIFEST is None:
    campaign_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    manifest_path = ROOT / "data" / "reference_results" / "phase4a_penalty_runs" / f"campaign_{campaign_id}.json"
    completed_rows = []
else:
    manifest_path = ROOT / RESUME_MANIFEST
    saved_campaign = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert saved_campaign["scope"] == "phase4a_common_penalty_pilot"
    assert saved_campaign["design_hash"] == DESIGN_HASH, "Cấu hình hoặc input đã khác đợt cần tiếp tục."
    campaign_id = saved_campaign["campaign_id"]
    completed_rows = saved_campaign["completed_runs"]
    assert len({row["run_key"] for row in completed_rows}) == len(completed_rows)
    expected = {row["run_key"] for row in planned}
    for row in completed_rows:
        assert row["run_key"] in expected
        path = ROOT / row["report"]
        old = json.loads(path.read_text(encoding="utf-8"))
        assert old["status"] == "completed"
        assert old["penalty"] == row["penalty"]
        assert old["penalty_analysis"] == penalty_diagnostics(bundles[(row["instance_id"], row["penalty"])], old)
        assert old["experiment"]["campaign_id"] == campaign_id
        assert old["experiment"]["design_hash"] == DESIGN_HASH
        assert old["experiment"]["run_key"] == row["run_key"]
        assert old["instance_sha256"] == DESIGN["instance_sha256"][row["instance_id"]]
        current_versions = {
            "python_version": platform.python_version(), "numpy_version": np.__version__,
            "scipy_version": scipy.__version__, "qiskit_version": qiskit.__version__,
            "qiskit_aer_version": qiskit_aer.__version__,
            "qiskit_optimization_version": qiskit_optimization.__version__,
        }
        for key, version in current_versions.items():
            assert old[key] == version, f"Phiên bản {key} đã khác đợt cần tiếp tục."
        assert old["platform"] == platform.platform(), "Môi trường hệ điều hành đã khác đợt cần tiếp tục."
        del old

print("Đợt:", campaign_id)
print("Chỉ mục:", manifest_path.relative_to(ROOT).as_posix())
print("Hoàn tất / dự kiến:", len(completed_rows), "/", len(planned))

def save_campaign():
    completed_keys = {row["run_key"] for row in completed_rows}
    payload = {
        "schema_version": 1, "scope": "phase4a_common_penalty_pilot",
        "campaign_id": campaign_id, "design_hash": DESIGN_HASH, "design": DESIGN,
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "planned_run_count": len(planned), "completed_run_count": len(completed_rows),
        "planned_runs": [{k: row[k] for k in ["run_key", "instance_id", "penalty", "initial_point_id", "seed"]}
                         for row in planned],
        "completed_runs": completed_rows,
        "remaining_run_keys": [row["run_key"] for row in planned if row["run_key"] not in completed_keys],
    }
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = manifest_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
                         encoding="utf-8")
    temporary.replace(manifest_path)

Đợt: 20261005T100508165284Z
Chỉ mục: data/reference_results/phase4a_penalty_runs/campaign_20261005T100508165284Z.json
Hoàn tất / dự kiến: 0 / 30


## Bước 6 — Chạy các lượt tuần tự và lưu ngay

Mỗi cặp seed/góc dùng một QAOA mới, cùng ngân sách. Mỗi lượt hoàn tất được lưu theo định dạng `metrics` của notebook 05, bổ sung `experiment` để liên kết với đợt này. Thư mục báo cáo vẫn là `phase3_runs/`, giúp C dùng bộ đọc đã có.

Chỉ giữ các hàng tóm tắt trong RAM; mẫu đầy đủ nằm trên đĩa. Giải phóng đối tượng mỗi lượt để tránh giữ nhiều báo cáo/mạch trong bộ nhớ.

`completed` chưa bảo đảm có nghiệm hợp lệ; 0/1024 vẫn được lưu đúng. Nếu chạm giới hạn thời gian, dừng đợt, không lưu JSON của lượt đó. Giới hạn kiểm tra sau một lần đánh giá, không ngắt giữa mạch. Các lượt hoàn tất trước đó vẫn còn và có thể tiếp tục qua chỉ mục.

In [6]:
completed_keys = {row["run_key"] for row in completed_rows}
for trial in planned:
    if trial["run_key"] in completed_keys:
        print("Bỏ qua lượt đã hoàn tất:", trial["instance_id"], trial["initial_point_id"], trial["seed"])
        continue
    name = trial["instance_id"]
    p = trial["penalty"]
    bundle = bundles[(name, p)]
    print(f"START {name} / P={p} / {trial['initial_point_id']} / seed={trial['seed']}", flush=True)
    report = run_qaoa_experiment(bundle, trial["config"], references[(name, p)])
    if report["status"] != "completed":
        print("Dừng đợt:", report["stop_reason"], "Không lưu JSON lượt này.")
        del report
        gc.collect()
        break
    report["experiment"] = {
        "campaign_id": campaign_id, "design_hash": DESIGN_HASH,
        "run_key": trial["run_key"], "initial_point_id": trial["initial_point_id"],
        "purpose": "controlled_common_penalty_pilot", "penalty": p,
    }
    diagnostics_started = perf_counter()
    report["penalty_analysis"] = penalty_diagnostics(bundle, report)
    report["penalty_analysis_seconds"] = perf_counter() - diagnostics_started
    path = save_experiment_report(ROOT, report)
    metrics = report["metrics"]
    best = metrics["best_observed_valid"]
    opt_prob = metrics["optimal_placement_probability"]
    row = {
        "run_key": trial["run_key"], "instance_id": name,
        "initial_point_id": trial["initial_point_id"], "seed": trial["seed"],
        "report": path.relative_to(ROOT).as_posix(),
        "num_qubits": report["num_qubits"], "penalty": report["penalty"],
        "shots": report["config"]["shots"], "reps": report["config"]["reps"],
        "maxiter": report["config"]["maxiter"],
        "original_best_objective": report["reference"]["original_best_objective"],
        "observed_best_objective": None if best is None else best["objective"],
        "found_valid": best is not None,
        "found_optimal": None if opt_prob is None else bool(opt_prob > 0),
        "feasible_probability": metrics["feasible_probability"],
        "optimal_placement_probability": opt_prob,
        "qubo_ground_state_probability": metrics["qubo_ground_state_probability"],
        "final_sample_mean_qubo_energy": metrics["final_sample_mean_qubo_energy"],
        **report["penalty_analysis"],
        "penalty_analysis_seconds": report["penalty_analysis_seconds"],
        "elapsed_seconds": report["elapsed_seconds"],
        "postprocessing_seconds": report["postprocessing_seconds"],
        "cost_function_evals": report["cost_function_evals"],
        "process_peak_rss_mib": report["resources"]["process_peak_rss_mib"],
    }
    completed_rows.append(row)
    save_campaign()
    completed_keys.add(trial["run_key"])
    print(f"SAVED: {path.name}; hợp lệ={round(row['feasible_probability'] * row['shots'])}/{row['shots']}; "
          f"thời gian={row['elapsed_seconds']:.2f}s", flush=True)
    del report, metrics, best
    gc.collect()
print("Hoàn tất trong đợt:", len(completed_rows), "/", len(planned))

START p0_reference / P=1 / goc_cu / seed=42
  evaluation 1: mean E=252.435547, elapsed=0.6s
  evaluation 5: mean E=260.736328, elapsed=2.6s
  evaluation 10: mean E=242.896484, elapsed=5.2s
  evaluation 15: mean E=248.152344, elapsed=13.3s
  evaluation 20: mean E=230.557617, elapsed=19.2s
  evaluation 25: mean E=243.530273, elapsed=26.4s
  evaluation 30: mean E=235.224609, elapsed=28.8s
SAVED: p0_reference_p1_seed42_20261005T100547899430Z.json; hợp lệ=3/1024; thời gian=29.22s
START p0_reference / P=1 / goc_cu / seed=43
  evaluation 1: mean E=250.141602, elapsed=0.5s
  evaluation 5: mean E=273.375000, elapsed=2.4s
  evaluation 10: mean E=247.377930, elapsed=4.9s
  evaluation 15: mean E=312.532227, elapsed=7.4s
  evaluation 20: mean E=206.445312, elapsed=9.8s
  evaluation 25: mean E=205.140625, elapsed=12.4s
  evaluation 30: mean E=199.197266, elapsed=14.8s
SAVED: p0_reference_p1_seed43_20261005T100606378665Z.json; hợp lệ=2/1024; thời gian=16.72s
START p0_reference / P=1 / goc_cu / seed=4

## Bước 7 — Kết quả từng lượt và đồng hạng

unique_mode_valid=None nghĩa là nhiều chuỗi cùng tần suất cao nhất, không phải không có placement hợp lệ. dominant_bitstring_count cho biết số chuỗi đồng hạng; dominant_valid_bitstring_count cho biết bao nhiêu chuỗi trong nhóm đó hợp lệ. Giữ riêng kết quả best_observed_valid, không sửa các bit trước khi chấm.

In [7]:
results = pd.DataFrame(completed_rows)
if results.empty:
    print("Chưa có lượt hoàn tất.")
else:
    table = results[["instance_id", "penalty", "initial_point_id", "seed", "feasible_probability",
                     "optimal_placement_probability", "observed_best_objective", "dominant_bitstring_count",
                     "dominant_valid_bitstring_count", "unique_mode_valid", "mean_total_squared_error", "elapsed_seconds"]].copy()
    table["feasible_probability"] *= 100
    table["optimal_placement_probability"] = pd.to_numeric(table["optimal_placement_probability"]) * 100
    display(table.round(6))
    print("Hai cột probability ở bảng này đã đổi sang %. None/NaN ở unique_mode_valid là đồng hạng.")

,instance_id,penalty,initial_point_id,seed,feasible_probability,optimal_placement_probability,observed_best_objective,dominant_bitstring_count,dominant_valid_bitstring_count,unique_mode_valid,mean_total_squared_error,elapsed_seconds
0,p0_reference,1.0,goc_cu,42,0.292969,0.292969,2.0,1,0,False,229.504883,29.220701
1,p0_reference,1.0,goc_cu,43,0.195312,0.195312,2.0,1024,2,None,197.981445,16.724300
2,p0_reference,1.0,goc_cu,44,0.097656,0.097656,2.0,1024,1,None,234.912109,14.888238
3,p0_reference,1.0,gamma_nho,42,0.195312,0.195312,2.0,1024,2,None,223.377930,16.283242
4,p0_reference,1.0,gamma_nho,43,0.195312,0.195312,2.0,1024,2,None,228.682617,16.514886
5,p0_reference,1.0,gamma_nho,44,0.195312,0.195312,2.0,1024,2,None,221.859375,16.545130
6,p0_reference,2.0,goc_cu,42,0.097656,0.097656,2.0,1,0,False,218.747070,15.241360
7,p0_reference,2.0,goc_cu,43,0.195312,0.195312,2.0,1024,2,None,199.516602,18.510866
8,p0_reference,2.0,goc_cu,44,0.000000,0.000000,NaN,1024,0,None,232.430664,16.760098
9,p0_reference,2.0,gamma_nho,42,0.000000,0.000000,NaN,1024,0,None,238.199219,18.229996


Hai cột probability ở bảng này đã đổi sang %. None/NaN ở unique_mode_valid là đồng hạng.


## Bước 8 — So sánh P theo từng instance và khởi tạo

Không chọn P từ năng lượng thô: E=sum(y)+P×V, nên thang đo thay đổi theo P. So tỷ lệ hợp lệ/tối ưu, số lượt có nghiệm, thời gian và V trung bình. V bao gồm slack sai. Mỗi nhóm có ba seed; đây là khảo sát khám phá, chưa phải kết luận thống kê mạnh.

In [8]:
def optional_sum(series):
    return None if series.isna().all() else int(series.sum())

summary = pd.DataFrame()
if not results.empty:
    summary = results.groupby(["instance_id", "penalty", "initial_point_id"], sort=False).agg(
        completed_runs=("run_key", "size"),
        valid_runs=("found_valid", "sum"),
        optimal_runs=("found_optimal", optional_sum),
        feasible_pct_mean=("feasible_probability", lambda s: 100 * s.mean()),
        feasible_pct_std=("feasible_probability", lambda s: 100 * s.std(ddof=1)),
        optimal_pct_mean=("optimal_placement_probability", lambda s: 100 * pd.to_numeric(s).mean()),
        mean_qubo_energy=("final_sample_mean_qubo_energy", "mean"),
        mean_squared_error=("mean_total_squared_error", "mean"),
        mean_assignment_error=("mean_assignment_error", "mean"),
        mean_capacity_equation_error=("mean_capacity_equation_error", "mean"),
        mean_solver_seconds=("elapsed_seconds", "mean"),
    ).reset_index()
    summary["planned_runs"] = len(SEEDS)
    summary["uniform_feasible_pct"] = summary.apply(lambda row: 100 * references[(row["instance_id"], row["penalty"])]["uniform_baseline"]["feasible_probability"], axis=1)
    display(summary.round(6))
    print("Các cột *_pct là phần trăm; *_std là độ lệch chuẩn giữa các lượt.")
    print("optimal_runs/optimal_pct_mean = NaN/N/A cho bộ vô nghiệm.")
    print("Chỉ kết luận cho cấu hình và các lượt thực sự có trong đợt này.")
    print("Năng lượng thô chỉ so trực tiếp trong cùng P. mean_squared_error là V, không nhân P.")

,instance_id,penalty,initial_point_id,completed_runs,valid_runs,optimal_runs,feasible_pct_mean,feasible_pct_std,optimal_pct_mean,mean_qubo_energy,mean_squared_error,mean_assignment_error,mean_capacity_equation_error,mean_solver_seconds,planned_runs,uniform_feasible_pct
0,p0_reference,1.0,goc_cu,3,3,3,0.195312,0.097656,0.195312,221.874674,220.799479,2.001302,218.798177,20.277746,3,0.097656
1,p0_reference,1.0,gamma_nho,3,3,3,0.195312,0.000000,0.195312,225.656250,224.639974,2.008464,222.631510,16.447753,3,0.097656
2,p0_reference,2.0,goc_cu,3,2,2,0.097656,0.097656,0.097656,434.857747,216.898112,1.880208,215.017904,16.837441,3,0.097656
3,p0_reference,2.0,gamma_nho,3,2,2,0.097656,0.097656,0.097656,473.061849,236.019857,1.998698,234.021159,17.052116,3,0.097656
4,p0_reference,3.0,goc_cu,3,0,0,0.000000,0.000000,0.000000,657.984375,218.967122,2.135417,216.831706,15.783370,3,0.097656
5,p0_reference,3.0,gamma_nho,3,1,1,0.032552,0.056382,0.032552,591.646810,196.837891,2.180339,194.657552,16.196561,3,0.097656
6,p0_reference,5.0,goc_cu,3,3,3,0.130208,0.056382,0.130208,1074.950846,214.778971,2.190755,212.588216,16.522982,3,0.097656
7,p0_reference,5.0,gamma_nho,3,3,3,0.162760,0.112764,0.162760,1067.933919,213.362956,1.866211,211.496745,15.102085,3,0.097656
8,p0_reference,8.0,goc_cu,3,3,3,0.130208,0.056382,0.130208,1845.108073,230.509440,2.014323,228.495117,15.496155,3,0.097656
9,p0_reference,8.0,gamma_nho,3,1,1,0.032552,0.056382,0.032552,1859.359375,232.290039,2.001302,230.288737,15.449795,3,0.097656


Các cột *_pct là phần trăm; *_std là độ lệch chuẩn giữa các lượt.
optimal_runs/optimal_pct_mean = NaN/N/A cho bộ vô nghiệm.
Chỉ kết luận cho cấu hình và các lượt thực sự có trong đợt này.
Năng lượng thô chỉ so trực tiếp trong cùng P. mean_squared_error là V, không nhân P.


## Bước 9 — Kiểm tra mẫu thô của một lượt

Chọn chỉ số hàng trong `completed_rows`. Bảng dưới giữ thứ tự tần suất từ báo cáo; mẫu đứng đầu có thể vi phạm. `best_observed_valid` là một cách đọc khác và phải báo cáo riêng nếu đề cương dùng mẫu phổ biến nhất.

Một placement hợp lệ có thể có slack sai, nên tỷ lệ placement hợp lệ/tối ưu và tỷ lệ cực tiểu QUBO khác nhau. Không sửa bit rồi gọi đó là mẫu đo thô.

In [9]:
INSPECT_RUN_INDEX = 0
if completed_rows:
    entry = completed_rows[INSPECT_RUN_INDEX]
    inspected = json.loads((ROOT / entry["report"]).read_text(encoding="utf-8"))
    samples = inspected["final_samples"]
    assert abs(sum(item["probability"] for item in samples) - 1) < 1e-6
    print("Nguồn:", entry["report"])
    print("P:", entry["penalty"])
    print("Đồng hạng và sai số:", inspected["penalty_analysis"])
    print("Khởi tạo:", entry["initial_point_id"], "seed:", entry["seed"])
    print("Nghiệm hợp lệ tốt nhất thực sự quan sát:", inspected["metrics"]["best_observed_valid"])
    display(pd.DataFrame([{
        "Xác suất": item["probability"], "Năng lượng": item["energy"],
        "Hợp lệ": item["decoded"]["feasible"], "Host": item["decoded"]["objective"],
        "x": item["decoded"]["x"], "y": item["decoded"]["y"],
    } for item in samples[:5]]))
    del inspected, samples
    gc.collect()

Nguồn: data/reference_results/phase3_runs/p0_reference_p1_seed42_20261005T100547899430Z.json
P: 1.0
Đồng hạng và sai số: {'mean_active_hosts': 1.052734375, 'mean_assignment_error': 2.0009765625, 'mean_capacity_equation_error': 227.50390625, 'mean_total_squared_error': 229.5048828125, 'reconstructed_mean_energy': 230.5576171875, 'dominant_bitstring_count': 1, 'dominant_valid_bitstring_count': 0, 'unique_mode_valid': False}
Khởi tạo: goc_cu seed: 42
Nghiệm hợp lệ tốt nhất thực sự quan sát: {'bits': [1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1], 'energy': 28.0, 'probability': 0.0009765625, 'feasible': True, 'objective': 2, 'one_hot_ok': True, 'capacity_ok': True, 'x': [[1, 0], [1, 0], [0, 1], [0, 1]], 'y': [1, 1], 'usage': [[5, 6], [5, 8]]}


,Xác suất,Năng lượng,Hợp lệ,Host,x,y
0,0.001953,347.0,False,None,"[[1, 1], [0, 1], [1, 1], [1, 1]]","[0, 1]"
1,0.000977,22.0,False,None,"[[1, 0], [0, 1], [1, 1], [0, 0]]","[1, 1]"
2,0.000977,93.0,False,None,"[[0, 0], [1, 0], [0, 0], [0, 1]]","[0, 1]"
3,0.000977,488.0,False,None,"[[1, 0], [1, 1], [1, 1], [1, 1]]","[1, 0]"
4,0.000977,365.0,False,None,"[[0, 1], [1, 1], [0, 0], [0, 1]]","[0, 0]"


## Bước 10 — Kiểm tra gói gửi C

Đối chiếu bảng tóm tắt với JSON nguồn để không nhầm lượt hoặc mất file. Báo cáo vẫn có `schema_version=2`, `metrics`, `reference`, `final_samples` như notebook 05; chỉ bổ sung `experiment`. Chỉ mục có schema riêng và không được cộng vào số lượt.

Gửi C notebook có đầu ra, chỉ mục và toàn bộ JSON được chỉ mục tham chiếu. Không yêu cầu C chạy lại QAOA.

In [10]:
if completed_rows:
    assert manifest_path.is_file()
    saved_campaign = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert saved_campaign["completed_run_count"] == len(completed_rows)
    assert len({item["run_key"] for item in completed_rows}) == len(completed_rows)
    for entry in completed_rows:
        source = json.loads((ROOT / entry["report"]).read_text(encoding="utf-8"))
        assert source["status"] == "completed"
        assert source["experiment"]["run_key"] == entry["run_key"]
        assert source["experiment"]["design_hash"] == DESIGN_HASH
        assert source["instance_sha256"] == DESIGN["instance_sha256"][entry["instance_id"]]
        for key in ["feasible_probability", "optimal_placement_probability",
                    "qubo_ground_state_probability"]:
            assert source["metrics"][key] == entry[key]
        assert source["penalty"] == entry["penalty"]
        diagnostics = penalty_diagnostics(bundles[(entry["instance_id"], entry["penalty"])], source)
        assert diagnostics == source["penalty_analysis"]
        assert all(entry[key] == value for key, value in diagnostics.items())
        del source
    print("PASS — các lượt duy nhất, báo cáo nguồn và chỉ mục khớp nhau.")
    print("Gửi C chỉ mục:", manifest_path.relative_to(ROOT).as_posix())
    print("Số hoàn tất / dự kiến:", len(completed_rows), "/", len(planned))
else:
    print("Chưa có lượt hoàn tất; không có báo cáo hoặc chỉ mục kết quả để bàn giao.")

PASS — các lượt duy nhất, báo cáo nguồn và chỉ mục khớp nhau.
Gửi C chỉ mục: data/reference_results/phase4a_penalty_runs/campaign_20261005T100508165284Z.json
Số hoàn tất / dự kiến: 30 / 30


## Sau đợt này

Lưu notebook có đầu ra, chỉ mục phase4a_penalty_runs và toàn bộ báo cáo nó tham chiếu. p0_reference: Restart Kernel, đặt RUN_INSTANCE_IDS=["p0_reference"], RESUME_MANIFEST=None, giữ các thông số khác; có 30 lượt. Không đổi thiết kế của đợt đang tiếp tục.

Đánh giá trước khi chọn P hoặc khảo sát hệ số gán/dung lượng riêng. Độ sâu p=1,2,3 là bước sau; không tăng p trong notebook này.

---

## English explanation

This notebook explores one shared penalty, A=B=P, following A1. It fixes depth p=1, 1024 shots, maxiter=35, initial points [0.3,0.7]/[0.3,0.05] and seeds 42/43/44. Penalties are 1,2,3,5,8. Default: tight/choice, 60 trials. Run reference separately, 30 trials. Independent penalties and depth sweeps remain future work.

Compute exact original and QUBO references for every penalty. Compare feasibility, original optimality and unweighted squared equation errors; raw energies have different scales across P. Errors include wrong slack. Keep every completed trial, including zero-feasibility outcomes. Record tied frequency maxima; unique_mode_valid is null when no unique mode exists.

Reports retain schema 2, adding penalty_analysis and its separate timing. The campaign index uses scope phase4a_common_penalty_pilot under phase4a_penalty_runs. Run keys include penalty. Persist completed trials immediately; resume only identical designs/inputs/versions/platforms. The time budget is checked between evaluations and memory peak covers process lifetime. Probe before full execution. Three seeds per group remain exploratory. Test-machine runs are not A's Windows data.